In [1]:

import json
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoConfig

SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


d:\SmolLM2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.15.0+cpu
Device: cpu


In [2]:

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

hf_config = AutoConfig.from_pretrained(MODEL_ID)

print("Model type:", hf_config.model_type)
print("Hidden size:", hf_config.hidden_size)
print("Layers:", hf_config.num_hidden_layers)
print("Attention heads:", hf_config.num_attention_heads)
print("KV heads:", hf_config.num_key_value_heads)
print("Intermediate size:", hf_config.intermediate_size)
print("Vocabulary size:", hf_config.vocab_size)
print("Tied embeddings:", hf_config.tie_word_embeddings)

Model type: llama
Hidden size: 576
Layers: 30
Attention heads: 9
KV heads: 3
Intermediate size: 1536
Vocabulary size: 49152
Tied embeddings: True


In [3]:

@dataclass
class SmolLMConfig:
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: int
    max_position_embeddings: int
    rms_norm_eps: float
    rope_theta: float
    tie_word_embeddings: bool
    attention_bias: bool = False
    mlp_bias: bool = False
    attention_dropout: float = 0.0

    @property
    def head_dim(self) -> int:
        assert self.hidden_size % self.num_attention_heads == 0
        return self.hidden_size // self.num_attention_heads

    def validate(self):
        assert self.vocab_size > 0
        assert self.hidden_size > 0
        assert self.num_hidden_layers > 0
        assert self.num_attention_heads > 0
        assert self.num_key_value_heads > 0

        assert self.hidden_size % self.num_attention_heads == 0
        assert self.num_attention_heads % self.num_key_value_heads == 0

        assert self.head_dim % 2 == 0
        assert self.intermediate_size > 0
        assert self.max_position_embeddings > 0
        assert self.rms_norm_eps > 0
        assert self.rope_theta > 0


config = SmolLMConfig(
    vocab_size=hf_config.vocab_size,
    hidden_size=hf_config.hidden_size,
    intermediate_size=hf_config.intermediate_size,
    num_hidden_layers=hf_config.num_hidden_layers,
    num_attention_heads=hf_config.num_attention_heads,
    num_key_value_heads=hf_config.num_key_value_heads,
    max_position_embeddings=hf_config.max_position_embeddings,
    rope_theta=getattr(hf_config, "rope_theta", 100000.0),
    rms_norm_eps=hf_config.rms_norm_eps,
    tie_word_embeddings=hf_config.tie_word_embeddings,
    attention_bias=hf_config.attention_bias,
    mlp_bias=getattr(hf_config, "mlp_bias", False),
    attention_dropout=hf_config.attention_dropout,
)

config.validate()
print(config)
print("Head dimension:", config.head_dim)


SmolLMConfig(vocab_size=49152, hidden_size=576, intermediate_size=1536, num_hidden_layers=30, num_attention_heads=9, num_key_value_heads=3, max_position_embeddings=8192, rms_norm_eps=1e-05, rope_theta=100000.0, tie_word_embeddings=True, attention_bias=False, mlp_bias=False, attention_dropout=0.0)
Head dimension: 64


In [4]:

class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype

        x = x.float()
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

        return (x * rms * self.weight.float()).to(input_dtype)

In [5]:
class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim: int,
        max_position_embeddings: int,
        theta: float = 100_000.0,
    ):
        super().__init__()

        assert head_dim % 2 == 0

        self.head_dim = head_dim
        self.theta = theta

        # Each pair of features uses its own rotation frequency.
        inv_freq = 1.0 / (
            theta ** (
                torch.arange(0, head_dim, 2, dtype=torch.float32)
                / head_dim
            )
        )

        self.register_buffer("inv_freq", inv_freq, persistent=False)

        positions = torch.arange(
            max_position_embeddings, dtype=torch.float32
        )

        freqs = torch.outer(positions, inv_freq)

        # Repeat each frequency for its corresponding feature pair.
        emb = torch.cat((freqs, freqs), dim=-1)

        self.register_buffer(
            "cos_cached", emb.cos(), persistent=False
        )
        self.register_buffer(
            "sin_cached", emb.sin(), persistent=False
        )

    def forward(self, x: torch.Tensor, position_ids=None):
        # x shape: (batch, heads, sequence_length, head_dim)
        seq_len = x.shape[-2]

        if position_ids is None:
            cos = self.cos_cached[:seq_len]
            sin = self.sin_cached[:seq_len]
        else:
            cos = self.cos_cached[position_ids]
            sin = self.sin_cached[position_ids]

        return cos, sin


In [6]:

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]

    return torch.cat((-x2, x1), dim=-1)



def apply_rope(q, k, cos, sin):
    # q: [batch, query_heads, seq_len, head_dim]
    # k: [batch, kv_heads, seq_len, head_dim]
    # cos/sin: [batch, seq_len, head_dim] or [seq_len, head_dim]

    if cos.ndim == 2:
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    # [batch, seq_len, head_dim] -> [batch, 1, seq_len, head_dim]
    cos = cos.unsqueeze(1).to(device=q.device, dtype=q.dtype)
    sin = sin.unsqueeze(1).to(device=q.device, dtype=q.dtype)

    q_rotated = q * cos + rotate_half(q) * sin

    # Match key dtype/device separately for robustness.
    cos_k = cos.to(device=k.device, dtype=k.dtype)
    sin_k = sin.to(device=k.device, dtype=k.dtype)
    k_rotated = k * cos_k + rotate_half(k) * sin_k

    return q_rotated, k_rotated



In [7]:

class GroupedQueryAttention(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.attention_dropout = config.attention_dropout

        # Query has more heads than key and value.
        self.q_proj = nn.Linear(
            self.hidden_size,
            self.num_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.k_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.v_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=config.attention_bias,
        )

        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim,
            self.hidden_size,
            bias=config.attention_bias,
        )

        self.rotary_emb = RotaryEmbedding(
            head_dim=self.head_dim,
            max_position_embeddings=config.max_position_embeddings,
            theta=config.rope_theta,
        )

    
    
    def forward(
        self,
        x: torch.Tensor,
        past_key_value=None,
        use_cache: bool = False,
        position_ids=None,
    ):
        batch_size, seq_len, _ = x.shape

        # 1. Project hidden states into Q, K, V.
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # 2. Reshape into attention heads.
        q = q.view(
            batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        # 3. Apply RoPE using absolute token positions.
        cos, sin = self.rotary_emb(q, position_ids)
        q, k = apply_rope(q, k, cos, sin)

        # 4. Append current K/V to the existing cache.
        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(2)

            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        # 5. Preserve the compact KV cache: 3 heads, not 9.
        present_key_value = (k, v) if use_cache else None

        # 6. Expand KV heads for grouped-query attention.
        k_attn = k.repeat_interleave(
            self.num_kv_groups, dim=1
        )
        v_attn = v.repeat_interleave(
            self.num_kv_groups, dim=1
        )

        # 7. Apply causal masking correctly.
        # Full prefill: prevent attending to future tokens.
        # Cached decoding: new tokens can attend to all cached
        # tokens and earlier tokens in the current input.
        if past_len == 0:
            is_causal = True
            attn_mask = None
        else:
            key_len = k_attn.size(2)
            query_positions = (
                past_len
                + torch.arange(seq_len, device=x.device)
            )
            key_positions = torch.arange(
                key_len, device=x.device
            )

            attn_mask = (
                key_positions.unsqueeze(0)
                <= query_positions.unsqueeze(1)
            )
            is_causal = False

        # 8. Compute scaled dot-product attention.
        attn_output = F.scaled_dot_product_attention(
            q,
            k_attn,
            v_attn,
            attn_mask=attn_mask,
            dropout_p=(
                self.attention_dropout if self.training else 0.0
            ),
            is_causal=is_causal,
        )

        # 9. Merge heads and project back to hidden size.
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(
            batch_size, seq_len, self.hidden_size
        )

        output = self.o_proj(attn_output)

        if use_cache:
            return output, present_key_value

        return output


In [8]:
class SwiGLU(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.gate_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=config.mlp_bias,
        )

        self.up_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=config.mlp_bias,
        )

        self.down_proj = nn.Linear(
            config.intermediate_size,
            config.hidden_size,
            bias=config.mlp_bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)

        hidden = gate * up

        return self.down_proj(hidden)


In [9]:
class DecoderBlock(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.input_layernorm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.self_attn = GroupedQueryAttention(config)

        self.post_attention_layernorm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        self.mlp = SwiGLU(config)

    
    def forward(
        self,
        x: torch.Tensor,
        past_key_value=None,
        use_cache: bool = False,
        position_ids=None,
    ):
        if use_cache:
            attn_output, present_key_value = self.self_attn(
                self.input_layernorm(x),
                past_key_value=past_key_value,
                use_cache=True,
                position_ids=position_ids,
            )
        else:
            attn_output = self.self_attn(
                self.input_layernorm(x),
                position_ids=position_ids,
            )

        x = x + attn_output
        x = x + self.mlp(self.post_attention_layernorm(x))

        if use_cache:
            return x, present_key_value

        return x


In [10]:

class SmolLMForCausalLM(nn.Module):
    def __init__(self, config: SmolLMConfig):
        super().__init__()

        self.config = config

        # 1. Convert token IDs into hidden vectors
        self.embed_tokens = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
        )

        # 2. Stack independent Transformer blocks
        self.layers = nn.ModuleList([
                DecoderBlock(config)
            for _ in range(config.num_hidden_layers)
        ])

        # 3. Normalize the final hidden representations
        self.norm = RMSNorm(
            config.hidden_size,
            eps=config.rms_norm_eps,
        )

        # 4. Project hidden vectors to vocabulary logits
        self.lm_head = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False,
        )

        # 5. Share input embedding and output projection weights
        if config.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

    
    def forward(
        self,
        input_ids: torch.Tensor,
        past_key_values=None,
        use_cache: bool = False,
    ):
        batch_size, seq_len = input_ids.shape

        # Each layer has its own cache. Determine the absolute position
        # of the first token in this input.
        past_length = 0
        if past_key_values is not None:
            past_length = past_key_values[0][0].shape[2]

        position_ids = torch.arange(
            past_length,
            past_length + seq_len,
            device=input_ids.device,
        ).unsqueeze(0).expand(batch_size, -1)

        x = self.embed_tokens(input_ids)
        next_cache = [] if use_cache else None

        for layer_idx, layer in enumerate(self.layers):
            past_key_value = (
                past_key_values[layer_idx]
                if past_key_values is not None
                else None
            )

            if use_cache:
                x, present_key_value = layer(
                    x,
                    past_key_value=past_key_value,
                    use_cache=True,
                    position_ids=position_ids,
                )
                next_cache.append(present_key_value)
            else:
                x = layer(x, position_ids=position_ids)

        x = self.norm(x)
        logits = self.lm_head(x)

        if use_cache:
            return logits, tuple(next_cache)

        return logits



In [11]:
from transformers import AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
hf_model.eval()

print(type(hf_model).__name__)
print("Reference parameters:", sum(p.numel() for p in hf_model.parameters()))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 527.25it/s]

LlamaForCausalLM
Reference parameters: 134515008


In [12]:

def load_hf_weights(
    model: SmolLMForCausalLM,
    hf_model: nn.Module,
):
    source_state = hf_model.state_dict()
    target_state = model.state_dict()

    mapped_state = {}
    used_source_keys = set()
    ignored_source_keys = set()

    for source_key, tensor in source_state.items():
        # Our RoPE computes these buffers locally.
        if source_key.endswith(
            ("rotary_emb.inv_freq",
             "rotary_emb.cos_cached",
             "rotary_emb.sin_cached")
        ):
            ignored_source_keys.add(source_key)
            continue

        if source_key.startswith("model.layers."):
            target_key = source_key.removeprefix("model.")
        elif source_key in (
            "model.embed_tokens.weight",
            "model.norm.weight",
        ):
            target_key = source_key.removeprefix("model.")
        elif source_key.startswith("lm_head."):
            target_key = source_key
        else:
            raise KeyError(
                f"Unrecognized HF parameter: {source_key}"
            )

        if target_key not in target_state:
            raise KeyError(
                f"No destination for {source_key} -> {target_key}"
            )

        if target_key in mapped_state:
            raise KeyError(f"Duplicate mapping: {target_key}")

        if tensor.shape != target_state[target_key].shape:
            raise ValueError(
                f"Shape mismatch for {target_key}: "
                f"HF={tuple(tensor.shape)}, "
                f"ours={tuple(target_state[target_key].shape)}"
            )

        mapped_state[target_key] = tensor
        used_source_keys.add(source_key)

    # If the checkpoint omits a tied output-head entry,
    # reuse its input embedding tensor.
    if model.config.tie_word_embeddings:
        embedding_key = "embed_tokens.weight"
        head_key = "lm_head.weight"

        if head_key not in mapped_state:
            mapped_state[head_key] = mapped_state[embedding_key]

    missing_keys = set(target_state) - set(mapped_state)
    if missing_keys:
        raise KeyError(f"Missing destination weights: {sorted(missing_keys)}")

    unexpected_keys = (
        set(source_state) - used_source_keys - ignored_source_keys
    )
    if unexpected_keys:
        raise KeyError(f"Unused HF weights: {sorted(unexpected_keys)}")

    model.load_state_dict(mapped_state, strict=True)

    # Verify that tied weights still share the same parameter.
    if model.config.tie_word_embeddings:
        assert (
            model.lm_head.weight is model.embed_tokens.weight
        )

    print(f"Mapped tensors: {len(used_source_keys)}")
    print(f"Ignored local RoPE buffers: {len(ignored_source_keys)}")
    print("All destination tensors loaded successfully.")


In [13]:
model = SmolLMForCausalLM(config)
load_hf_weights(model, hf_model)

model.eval()

Mapped tensors: 273
Ignored local RoPE buffers: 0
All destination tensors loaded successfully.


SmolLMForCausalLM(
  (embed_tokens): Embedding(49152, 576)
  (layers): ModuleList(
    (0-29): 30 x DecoderBlock(
      (input_layernorm): RMSNorm()
      (self_attn): GroupedQueryAttention(
        (q_proj): Linear(in_features=576, out_features=576, bias=False)
        (k_proj): Linear(in_features=576, out_features=192, bias=False)
        (v_proj): Linear(in_features=576, out_features=192, bias=False)
        (o_proj): Linear(in_features=576, out_features=576, bias=False)
        (rotary_emb): RotaryEmbedding()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): SwiGLU(
        (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
        (up_proj): Linear(in_features=576, out_features=1536, bias=False)
        (down_proj): Linear(in_features=1536, out_features=576, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=576, out_features=49152, bias=False)
)

In [14]:

import os
import json
import torch
from safetensors.torch import save_file

EXPORT_DIR = "./smollm2_hf_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Get custom model weights
state_dict = model.state_dict()

# 2. Convert parameter names to HF Llama format
hf_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("embed_tokens."):
        new_key = "model." + key

    elif key.startswith("layers."):
        new_key = "model." + key

    elif key.startswith("norm."):
        new_key = "model." + key

    elif key.startswith("lm_head."):
        new_key = key

    else:
        print("Skipping:", key)
        continue

    hf_state_dict[new_key] = value.detach().cpu().contiguous().clone()

# 3. Check tensor count and shapes
print("Original tensors:", len(state_dict))
print("Export tensors:", len(hf_state_dict))

# 4. Save weights
save_file(
    hf_state_dict,
    os.path.join(EXPORT_DIR, "model.safetensors")
)

print("Weights exported successfully!")


Original tensors: 273
Export tensors: 273
Weights exported successfully!


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)
print("Special tokens:", tokenizer.special_tokens_map)

Vocabulary size: 49152
Model max length: 8192
Special tokens: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}


In [16]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by a strictly positive temperature."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")

    return logits / temperature


def apply_top_k(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Keep the k highest logits and mask all remaining logits."""
    vocab_size = logits.shape[-1]

    if not 1 <= k <= vocab_size:
        raise ValueError(f"k must be between 1 and {vocab_size}")

    top_k_values, _ = torch.topk(logits, k, dim=-1)
    threshold = top_k_values[..., -1, None]

    return logits.masked_fill(logits < threshold, float("-inf"))

def apply_top_p(logits: torch.Tensor, top_p: float) -> torch.Tensor:
    """Keep the smallest high-probability token set reaching top_p."""
    if not 0.0 < top_p <= 1.0:
        raise ValueError("top_p must be greater than 0 and at most 1")

    sorted_logits, sorted_indices = torch.sort(
        logits, descending=True, dim=-1
    )

    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Initially remove tokens whose cumulative probability exceeds top_p.
    sorted_remove = cumulative_probs > top_p

    # Keep the token that crosses the threshold as well.
    sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
    sorted_remove[..., 0] = False

    # Restore the original vocabulary ordering.
    remove_mask = torch.zeros_like(sorted_remove)
    remove_mask.scatter_(-1, sorted_indices, sorted_remove)

    return logits.masked_fill(remove_mask, float("-inf"))


In [17]:
print("Temperature scaling:")
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

device = next(model.parameters()).device
input_ids = input_ids.to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

for temperature in (0.5, 1.0, 2.0):
    scaled_logits = apply_temperature(logits, temperature)
    probabilities = torch.softmax(scaled_logits, dim=-1)

    top_probs, top_ids = probabilities[0].topk(5)

    print(f"\nTemperature: {temperature}")
    for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
        token = tokenizer.decode([token_id])
        print(f"  {token!r:20} {probability:.6f}")

print("\nTop-k sampling:")
prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

k = 5
filtered_logits = apply_top_k(logits, k)
probabilities = torch.softmax(filtered_logits, dim=-1)

print("Tokens remaining:", torch.isfinite(filtered_logits).sum().item())
print("Probability sum:", probabilities.sum().item())

top_probs, top_ids = probabilities[0].topk(k)

for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20} {probability:.6f}")


print("\nTop-p sampling:")
prompt = "The capital of France is"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

model.eval()

with torch.inference_mode():
    logits = model(input_ids)[:, -1, :]

top_p = 0.9
filtered_logits = apply_top_p(logits, top_p)
probabilities = torch.softmax(filtered_logits, dim=-1)

print("Tokens retained:", torch.isfinite(filtered_logits).sum().item())
print("Probability sum:", probabilities.sum().item())

top_probs, top_ids = probabilities[0].topk(5)

for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"{tokenizer.decode([token_id])!r:20} {probability:.6f}")




Temperature scaling:

Temperature: 0.5
  ' the'               0.771823
  ' Paris'             0.099259
  ' located'           0.060194
  ' called'            0.021693
  ' a'                 0.017330

Temperature: 1.0
  ' the'               0.261682
  ' Paris'             0.093842
  ' located'           0.073079
  ' called'            0.043871
  ' a'                 0.039211

Temperature: 2.0
  ' the'               0.012274
  ' Paris'             0.007350
  ' located'           0.006486
  ' called'            0.005026
  ' a'                 0.004751

Top-k sampling:
Tokens remaining: 5
Probability sum: 1.0
' the'               0.511412
' Paris'             0.183399
' located'           0.142820
' called'            0.085738
' a'                 0.076631

Top-p sampling:
Tokens retained: 406
Probability sum: 1.0
' the'               0.290708
' Paris'             0.104252
' located'           0.081185
' called'            0.048737
' a'                 0.043560


In [18]:

import torch

@torch.inference_mode()
def generate_sample(
    model,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    eos_token_id: int | None = None,
    seed: int | None = None,
) -> torch.Tensor:
    """Generate tokens using temperature-scaled, filtered sampling."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens cannot be negative")

    device = input_ids.device
    generator = None

    if seed is not None:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

    model.eval()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        logits = model(generated)[:, -1, :]
        logits = apply_temperature(logits, temperature)

        if top_k is not None:
            logits = apply_top_k(logits, top_k)

        if top_p is not None:
            logits = apply_top_p(logits, top_p)

        probabilities = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        )

        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None:
            if (next_token == eos_token_id).all():
                break

    return generated


In [19]:

@torch.inference_mode()
def generate_sample_cached(
    model,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
    eos_token_id: int | None = None,
    seed: int | None = None,
) -> torch.Tensor:
    """Sample tokens while reusing each layer's KV cache."""
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens cannot be negative")

    device = input_ids.device
    generator = None

    if seed is not None:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

    model.eval()
    generated = input_ids.clone()

    if max_new_tokens == 0:
        return generated

    # Prefill: process the entire prompt once.
    logits, past_key_values = model(
        input_ids,
        use_cache=True,
    )

    for step in range(max_new_tokens):
        next_logits = logits[:, -1, :]
        next_logits = apply_temperature(next_logits, temperature)

        if top_k is not None:
            next_logits = apply_top_k(next_logits, top_k)

        if top_p is not None:
            next_logits = apply_top_p(next_logits, top_p)

        probabilities = torch.softmax(next_logits, dim=-1)

        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        )

        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None and (next_token == eos_token_id).all():
            break

        # The final sampled token needs no forward pass if generation ends.
        if step == max_new_tokens - 1:
            break

        # Decode only the newly sampled token, reusing the K/V cache.
        logits, past_key_values = model(
            next_token,
            past_key_values=past_key_values,
            use_cache=True,
        )

    return generated


In [37]:

prompt = "What is the model name"
device = next(model.parameters()).device

input_ids = tokenizer(
    prompt, return_tensors="pt"
)["input_ids"].to(device)

generation_args = {
    "max_new_tokens": 30,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.9,
    "eos_token_id": tokenizer.eos_token_id,
    "seed": 42,
}

uncached_output = generate_sample(
    model, input_ids, **generation_args
)

cached_output = generate_sample_cached(
    model, input_ids, **generation_args
)

print(
    "Exact token match:",
    torch.equal(uncached_output, cached_output),
)

print("\nUncached:")
print(tokenizer.decode(uncached_output[0], skip_special_tokens=True))

print("\nCached:")
print(tokenizer.decode(cached_output[0], skip_special_tokens=True))


Exact token match: True

Uncached:
What is the model name? The model name is the name of the model that you will be using to create your model. If you have a model that is called "log

Cached:
What is the model name? The model name is the name of the model that you will be using to create your model. If you have a model that is called "log


In [21]:

import json
from pathlib import Path

DATA_DIR = Path("./dummy_corpus")  # Folder containing smollm.ipynb and the JSONL files

DATA_FILES = {
    "train": DATA_DIR / "train.jsonl",
    "val": DATA_DIR / "val.jsonl",
    "test": DATA_DIR / "test.jsonl",
    "general_eval": DATA_DIR / "general_eval.jsonl",
}

# Confirm the notebook can find all four files.
for split, path in DATA_FILES.items():
    print(f"{split:12} | exists={path.exists()} | {path.resolve()}")

missing = [str(p) for p in DATA_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Could not find these files:\n" + "\n".join(missing)
    )


train        | exists=True | D:\SmolLM2\dummy_corpus\train.jsonl
val          | exists=True | D:\SmolLM2\dummy_corpus\val.jsonl
test         | exists=True | D:\SmolLM2\dummy_corpus\test.jsonl
general_eval | exists=True | D:\SmolLM2\dummy_corpus\general_eval.jsonl


In [22]:

def clean_text(text: str) -> str:
    """Normalize line endings and remove invalid null bytes."""
    return (
        text.replace("\r\n", "\n")
            .replace("\r", "\n")
            .replace("\x00", "")
            .strip()
    )


def load_jsonl(path: Path) -> list[dict]:
    documents = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                doc = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_number}"
                ) from exc

            required_fields = {"id", "kind", "domain", "title", "text"}
            missing_fields = required_fields - doc.keys()

            if missing_fields:
                raise ValueError(
                    f"{path}, line {line_number}: missing "
                    f"{sorted(missing_fields)}"
                )

            if not isinstance(doc["text"], str):
                raise TypeError(
                    f"{path}, line {line_number}: text must be a string"
                )

            # Preserve the original metadata and clean text only.
            doc = doc.copy()
            doc["title"] = str(doc["title"]).strip()
            doc["text"] = clean_text(doc["text"])

            if not doc["text"]:
                raise ValueError(
                    f"{path}, line {line_number}: empty text after cleaning"
                )

            documents.append(doc)

    return documents


corpus = {
    split: load_jsonl(path)
    for split, path in DATA_FILES.items()
}

for split, documents in corpus.items():
    total_chars = sum(len(doc["text"]) for doc in documents)
    print(
        f"{split:12} | documents={len(documents):5,} "
        f"| characters={total_chars:,}"
    )


train        | documents=8,859 | characters=7,193,229
val          | documents=  492 | characters=409,282
test         | documents=  492 | characters=398,761
general_eval | documents=  300 | characters=183,188


In [23]:

for split in ("train", "val", "test"):
    print(f"\n{'=' * 20} {split.upper()} {'=' * 20}")

    for doc in corpus[split][:2]:
        print(f"\nID:     {doc['id']}")
        print(f"Kind:   {doc['kind']}")
        print(f"Domain: {doc['domain']}")
        print(f"Title:  {doc['title']}")
        print(f"Text:\n{doc['text'][:700]}")



==================== TRAIN ====================

ID:     doc-004259
Kind:   concept
Domain: social network
Title:  Understanding the variable-length path in graph databases
Text:
A variable-length path matches a chain of relationships whose length lies between a minimum and a maximum. It expresses ideas like friends within three hops without writing three separate patterns.

Consider a social network model built from Person, Post, Group. To work out who are the friends of friends of a user, you rely directly on the variable-length path.

A frequent problem: Leaving out the upper bound allows the search to grow with the size of the graph.

It is closely related to the query parameter. A query parameter is a named placeholder whose value is supplied separately from the query text.

Recommended practice: Always write an explicit upper bound such as 1..4.

ID:     doc-007949
Kind:   cypher
Domain: social network
Title:  Cypher example: find how closely Asha and Elena are connected in the 

In [24]:

import hashlib
from collections import Counter

# 1. Check duplicate document IDs within each split
for split in ("train", "val", "test", "general_eval"):
    ids = [doc["id"] for doc in corpus[split]]
    duplicate_ids = [doc_id for doc_id, count in Counter(ids).items() if count > 1]

    print(
        f"{split:12} | unique IDs={len(set(ids)):,} "
        f"| duplicate IDs={len(duplicate_ids)}"
    )

    if duplicate_ids:
        raise ValueError(f"Duplicate document IDs in {split}: {duplicate_ids[:5]}")


# 2. Check ID and exact-text overlap between train, validation, and test
def text_hash(doc):
    return hashlib.sha256(doc["text"].encode("utf-8")).hexdigest()

split_names = ("train", "val", "test")

id_sets = {
    split: {doc["id"] for doc in corpus[split]}
    for split in split_names
}

text_hash_sets = {
    split: {text_hash(doc) for doc in corpus[split]}
    for split in split_names
}

for i, split_a in enumerate(split_names):
    for split_b in split_names[i + 1:]:
        shared_ids = id_sets[split_a] & id_sets[split_b]
        shared_texts = text_hash_sets[split_a] & text_hash_sets[split_b]

        print(
            f"{split_a} vs {split_b}: "
            f"shared IDs={len(shared_ids)}, "
            f"identical texts={len(shared_texts)}"
        )

        if shared_ids or shared_texts:
            raise ValueError(
                f"Potential data leakage between {split_a} and {split_b}"
            )

print("\nD2 passed: no duplicate IDs within splits and no ID/text overlap across train, val, and test.")


train        | unique IDs=8,859 | duplicate IDs=0
val          | unique IDs=492 | duplicate IDs=0
test         | unique IDs=492 | duplicate IDs=0
general_eval | unique IDs=300 | duplicate IDs=0
train vs val: shared IDs=0, identical texts=0
train vs test: shared IDs=0, identical texts=0
val vs test: shared IDs=0, identical texts=0

D2 passed: no duplicate IDs within splits and no ID/text overlap across train, val, and test.


In [25]:

BLOCK_SIZE = 512

EOS_TOKEN_ID = tokenizer.eos_token_id

assert EOS_TOKEN_ID is not None, "Tokenizer must define an EOS token."
assert BLOCK_SIZE > 1

print("Block size:", BLOCK_SIZE)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", EOS_TOKEN_ID)


Block size: 512
EOS token: <|endoftext|>
EOS token ID: 0


In [26]:

def tokenize_and_pack(documents, tokenizer, block_size=512):
    token_stream = []

    for doc in documents:
        content = f"{doc['title']}\n\n{doc['text']}"
        token_ids = tokenizer.encode(
            content,
            add_special_tokens=False
        )

        # Preserve a boundary between consecutive documents.
        token_stream.extend(token_ids)
        token_stream.append(tokenizer.eos_token_id)

    # Keep only complete blocks; discard the short remainder.
    usable_length = (len(token_stream) // (block_size + 1)) * (block_size + 1)
    token_stream = token_stream[:usable_length]

    if not token_stream:
        raise ValueError("No complete token blocks were produced.")

    packed = torch.tensor(token_stream, dtype=torch.long)
    packed = packed.view(-1, block_size + 1)

    input_ids = packed[:, :-1].contiguous()
    targets = packed[:, 1:].contiguous()

    return {
        "input_ids": input_ids,
        "targets": targets,
        "token_count": usable_length,
        "block_size": block_size,
    }


packed_data = {}

for split in ("train", "val", "test", "general_eval"):
    packed_data[split] = tokenize_and_pack(
        corpus[split],
        tokenizer,
        block_size=BLOCK_SIZE,
    )

    data = packed_data[split]
    print(
        f"{split:12} | "
        f"tokens={data['token_count']:,} | "
        f"blocks={data['input_ids'].shape[0]:,} | "
        f"input shape={tuple(data['input_ids'].shape)} | "
        f"target shape={tuple(data['targets'].shape)}"
    )


train        | tokens=1,833,462 | blocks=3,574 | input shape=(3574, 512) | target shape=(3574, 512)
val          | tokens=103,626 | blocks=202 | input shape=(202, 512) | target shape=(202, 512)
test         | tokens=101,574 | blocks=198 | input shape=(198, 512) | target shape=(198, 512)
general_eval | tokens=40,014 | blocks=78 | input shape=(78, 512) | target shape=(78, 512)


In [27]:

train_data = packed_data["train"]

assert train_data["input_ids"].shape == train_data["targets"].shape
assert train_data["input_ids"].shape[1] == BLOCK_SIZE
assert train_data["input_ids"].dtype == torch.long
assert train_data["targets"].dtype == torch.long

# Causal language modeling: targets must be inputs shifted by one token.
first_block = train_data["input_ids"][0]
first_targets = train_data["targets"][0]

assert torch.equal(first_block[1:], first_targets[:-1])

print("D3 passed: tokenization, packing, and next-token target alignment are valid.")
print("First input:", tokenizer.decode(first_block[:40].tolist()))
print("First target:", tokenizer.decode(first_targets[:40].tolist()))


D3 passed: tokenization, packing, and next-token target alignment are valid.
First input: Understanding the variable-length path in graph databases

A variable-length path matches a chain of relationships whose length lies between a minimum and a maximum. It expresses ideas like friends within three hops without
First target:  the variable-length path in graph databases

A variable-length path matches a chain of relationships whose length lies between a minimum and a maximum. It expresses ideas like friends within three hops without writing


In [28]:

from torch.utils.data import Dataset, DataLoader

class PackedLanguageModelDataset(Dataset):
    def __init__(self, packed_split):
        self.input_ids = packed_split["input_ids"]
        self.targets = packed_split["targets"]

        assert self.input_ids.shape == self.targets.shape

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, index):
        return {
            "input_ids": self.input_ids[index],
            "targets": self.targets[index],
        }


datasets = {
    split: PackedLanguageModelDataset(data)
    for split, data in packed_data.items()
}

for split, dataset in datasets.items():
    print(f"{split:12} | sequences={len(dataset):,}")


train        | sequences=3,574
val          | sequences=202
test         | sequences=198
general_eval | sequences=78


In [29]:

BATCH_SIZE = 2
NUM_WORKERS = 0  # Reliable default for Windows/Jupyter
PIN_MEMORY = torch.cuda.is_available()

loaders = {
    split: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=(split == "train"),
    )
    for split, dataset in datasets.items()
}

print("Batch size:", BATCH_SIZE)
print("Pin memory:", PIN_MEMORY)
print("Training batches:", len(loaders["train"]))
print("Validation batches:", len(loaders["val"]))
print("Test batches:", len(loaders["test"]))


Batch size: 2
Pin memory: False
Training batches: 1787
Validation batches: 101
Test batches: 99


In [30]:

batch = next(iter(loaders["train"]))

input_ids = batch["input_ids"]
targets = batch["targets"]

print("Input shape:", input_ids.shape)
print("Target shape:", targets.shape)
print("Input dtype:", input_ids.dtype)
print("Target dtype:", targets.dtype)
print("Token ID range:", input_ids.min().item(), "to", input_ids.max().item())
print("Vocabulary size:", config.vocab_size)

assert input_ids.shape == (BATCH_SIZE, BLOCK_SIZE)
assert targets.shape == input_ids.shape
assert input_ids.dtype == torch.long
assert targets.dtype == torch.long
assert input_ids.min() >= 0 and input_ids.max() < config.vocab_size
assert targets.min() >= 0 and targets.max() < config.vocab_size

print("D4 passed: DataLoader produces valid causal-LM batches.")

Input shape: torch.Size([2, 512])
Target shape: torch.Size([2, 512])
Input dtype: torch.int64
Target dtype: torch.int64
Token ID range: 0 to 48687
Vocabulary size: 49152
D4 passed: DataLoader produces valid causal-LM batches.


In [31]:

# # ============================================================
# # PART E — SMOLLM2 CONTINUED PRETRAINING
# # E1 Loss | E2 Baseline | E3 Schedule | E4 Training
# # E5 Accumulation/Precision | E6 Validation | E7 Resume
# # ============================================================

# import copy
# import math
# import random
# from pathlib import Path

# import numpy as np
# import matplotlib.pyplot as plt

# import torch
# import torch.nn.functional as F


# # ------------------------------------------------------------
# # E0. CONFIGURATION
# # ------------------------------------------------------------

# SEED = 42

# NUM_EPOCHS = 3
# LEARNING_RATE = 2e-5
# WEIGHT_DECAY = 0.01
# WARMUP_RATIO = 0.05
# MIN_LR_RATIO = 0.10

# GRAD_ACCUM_STEPS = 4
# MAX_GRAD_NORM = 1.0

# # Start conservatively; adjust after observing memory usage.
# TRAIN_BATCH_SIZE = BATCH_SIZE
# EVAL_BATCH_SIZE = 1

# # Choose "auto", "fp32", "fp16", or "bf16".
# PRECISION = "auto"

# # None means evaluate the entire validation split.
# MAX_EVAL_BATCHES = None

# # Checkpoint after every N optimizer updates.
# SAVE_EVERY_STEPS = 50
# EVAL_EVERY_STEPS = 50

# CHECKPOINT_DIR = Path("checkpoints_smollm2")
# CHECKPOINT_PATH = CHECKPOINT_DIR / "continued_pretraining.pt"

# # Set True to resume from the latest checkpoint if present.
# RESUME_IF_AVAILABLE = True

# # Small diagnostic: repeatedly train on one batch, then restore
# # the original pretrained weights.
# SMOKE_TEST_STEPS = 5


# # ------------------------------------------------------------
# # E0.1 DEVICE, SEEDS, AND PRECISION
# # ------------------------------------------------------------

# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)

# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# if PRECISION == "auto":
#     if device.type == "cuda" and torch.cuda.is_bf16_supported():
#         PRECISION = "bf16"
#     elif device.type == "cuda":
#         PRECISION = "fp16"
#     else:
#         PRECISION = "fp32"

# if PRECISION == "bf16":
#     assert device.type == "cuda" and torch.cuda.is_bf16_supported(), \
#         "BF16 was selected but is not supported by this device."

# if PRECISION == "fp16":
#     assert device.type == "cuda", \
#         "FP16 mixed precision is configured here for CUDA."

# AMP_DTYPE = {
#     "fp32": torch.float32,
#     "fp16": torch.float16,
#     "bf16": torch.bfloat16,
# }[PRECISION]

# USE_AMP = PRECISION != "fp32"
# USE_SCALER = PRECISION == "fp16"

# # Use the modern API where available.
# try:
#     scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
# except (AttributeError, TypeError):
#     scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)

# print("Device:", device)
# print("Precision:", PRECISION)
# print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
# print("Train batch size:", TRAIN_BATCH_SIZE)
# print("Gradient accumulation:", GRAD_ACCUM_STEPS)
# print("Effective batch size:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)


# def autocast_context():
#     if not USE_AMP:
#         return torch.autocast(device_type=device.type, enabled=False)

#     return torch.autocast(
#         device_type=device.type,
#         dtype=AMP_DTYPE,
#         enabled=True,
#     )


# # ------------------------------------------------------------
# # E1. NEXT-TOKEN CROSS-ENTROPY
# # ------------------------------------------------------------

# def causal_lm_loss(logits, targets):
#     """
#     logits:  [batch, sequence, vocabulary]
#     targets: [batch, sequence]

#     The dataset has already shifted targets by one token.
#     """
#     return F.cross_entropy(
#         logits.reshape(-1, logits.size(-1)).float(),
#         targets.reshape(-1),
#     )


# def forward_logits(input_ids):
#     output = model(input_ids)

#     # Support either logits-only or (logits, cache) returns.
#     if isinstance(output, tuple):
#         output = output[0]

#     return output


# # ------------------------------------------------------------
# # E2. EVALUATION / PRETRAINED BASELINE
# # ------------------------------------------------------------

# @torch.no_grad()
# def evaluate(split, max_batches=None):
#     model.eval()

#     inputs = packed_data[split]["input_ids"]
#     targets = packed_data[split]["targets"]

#     total_loss = 0.0
#     total_tokens = 0
#     num_examples = inputs.size(0)
#     num_batches = math.ceil(num_examples / EVAL_BATCH_SIZE)

#     if max_batches is not None:
#         num_batches = min(num_batches, max_batches)

#     for batch_idx in range(num_batches):
#         start = batch_idx * EVAL_BATCH_SIZE
#         end = min(start + EVAL_BATCH_SIZE, num_examples)

#         x = inputs[start:end].to(device)
#         y = targets[start:end].to(device)

#         with autocast_context():
#             logits = forward_logits(x)
#             loss = causal_lm_loss(logits, y)

#         token_count = y.numel()
#         total_loss += loss.item() * token_count
#         total_tokens += token_count

#     if total_tokens == 0:
#         raise ValueError(f"No evaluation tokens processed for {split}.")

#     mean_loss = total_loss / total_tokens
#     perplexity = math.exp(min(mean_loss, 80.0))

#     model.train()

#     return {
#         "loss": mean_loss,
#         "perplexity": perplexity,
#         "tokens": total_tokens,
#     }


# # Record the pretrained model's quality BEFORE any updates.
# baseline = evaluate("val", MAX_EVAL_BATCHES)

# print("\nPretrained baseline on validation data")
# print(f"Validation loss:       {baseline['loss']:.4f}")
# print(f"Validation perplexity: {baseline['perplexity']:.4f}")
# print(f"Tokens evaluated:      {baseline['tokens']:,}")


# # ------------------------------------------------------------
# # E4. TINY OVERFIT SMOKE TEST
# # ------------------------------------------------------------

# # Preserve the exact pretrained parameters: the diagnostic must
# # not affect the real training run.
# original_state = copy.deepcopy(model.state_dict())

# smoke_optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-5,
#     weight_decay=0.0,
# )

# smoke_batch = next(iter(loaders["train"]))
# smoke_x = smoke_batch["input_ids"].to(device)
# smoke_y = smoke_batch["targets"].to(device)

# smoke_losses = []

# model.train()

# for _ in range(SMOKE_TEST_STEPS):
#     smoke_optimizer.zero_grad(set_to_none=True)

#     with autocast_context():
#         smoke_logits = forward_logits(smoke_x)
#         smoke_loss = causal_lm_loss(smoke_logits, smoke_y)

#     if not torch.isfinite(smoke_loss):
#         raise RuntimeError("Non-finite loss during smoke test.")

#     smoke_loss.backward()
#     torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
#     smoke_optimizer.step()

#     smoke_losses.append(smoke_loss.item())

# # Restore all original pretrained parameters.
# model.load_state_dict(original_state)
# del original_state, smoke_optimizer, smoke_logits, smoke_loss
# model.train()

# print("\nTiny overfit smoke test")
# print("Losses:", [round(x, 4) for x in smoke_losses])
# print(
#     "First-to-last loss change:",
#     round(smoke_losses[0] - smoke_losses[-1], 4),
# )
# print("Original pretrained weights restored.")

# if not all(math.isfinite(x) for x in smoke_losses):
#     raise RuntimeError("Smoke test produced non-finite losses.")

# # A falling trend is encouraging, but five updates are too few
# # to guarantee strict monotonic decrease.
# if smoke_losses[-1] >= smoke_losses[0]:
#     print(
#         "Note: loss did not fall over this short test. "
#         "Inspect the losses before committing to a long run."
#     )


# # ------------------------------------------------------------
# # E3. ADAMW + WARMUP / COSINE DECAY
# # ------------------------------------------------------------

# train_examples = len(datasets["train"])
# batches_per_epoch = train_examples // TRAIN_BATCH_SIZE

# if batches_per_epoch == 0:
#     raise ValueError("Training set is smaller than one training batch.")

# updates_per_epoch = math.ceil(batches_per_epoch / GRAD_ACCUM_STEPS)
# TOTAL_STEPS = NUM_EPOCHS * updates_per_epoch
# WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)

# def lr_multiplier(step):
#     if WARMUP_STEPS > 0 and step < WARMUP_STEPS:
#         return max((step + 1) / WARMUP_STEPS, 1e-8)

#     decay_steps = max(TOTAL_STEPS - WARMUP_STEPS, 1)
#     progress = min(
#         max((step - WARMUP_STEPS) / decay_steps, 0.0),
#         1.0,
#     )
#     cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

#     return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine


# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     weight_decay=WEIGHT_DECAY,
# )

# scheduler = torch.optim.lr_scheduler.LambdaLR(
#     optimizer,
#     lr_lambda=lr_multiplier,
# )

# # Visualize the learning-rate schedule before training.
# schedule_lrs = [
#     LEARNING_RATE * lr_multiplier(step)
#     for step in range(TOTAL_STEPS)
# ]

# plt.figure(figsize=(9, 4))
# plt.plot(range(TOTAL_STEPS), schedule_lrs)
# plt.title("Warmup + Cosine Learning-Rate Schedule")
# plt.xlabel("Optimizer step")
# plt.ylabel("Learning rate")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# print("Batches per epoch:", batches_per_epoch)
# print("Optimizer updates per epoch:", updates_per_epoch)
# print("Planned total optimizer updates:", TOTAL_STEPS)
# print("Warmup updates:", WARMUP_STEPS)


# # ------------------------------------------------------------
# # E5 / E7. EXACT CHECKPOINT STATE FOR STEP-LEVEL RESUME
# # ------------------------------------------------------------

# CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# history = {
#     "train_steps": [],
#     "train_loss": [],
#     "val_steps": [],
#     "val_loss": [],
#     "val_perplexity": [],
#     "lr_steps": [],
#     "lr": [],
# }

# global_step = 0
# start_epoch = 0
# batch_cursor = 0
# epoch_permutation = None

# # Dedicated generator makes the next epoch's permutation reproducible.
# shuffle_generator = torch.Generator(device="cpu")
# shuffle_generator.manual_seed(SEED)


# def save_checkpoint():
#     state = {
#         "model": model.state_dict(),
#         "optimizer": optimizer.state_dict(),
#         "scheduler": scheduler.state_dict(),
#         "scaler": scaler.state_dict(),
#         "global_step": global_step,
#         "epoch": start_epoch,
#         "batch_cursor": batch_cursor,
#         "epoch_permutation": epoch_permutation,
#         "shuffle_generator": shuffle_generator.get_state(),
#         "torch_rng": torch.get_rng_state(),
#         "numpy_rng": np.random.get_state(),
#         "python_rng": random.getstate(),
#         "cuda_rng": (
#             torch.cuda.get_rng_state_all()
#             if torch.cuda.is_available()
#             else None
#         ),
#         "history": history,
#         "baseline": baseline,
#         "precision": PRECISION,
#         "block_size": BLOCK_SIZE,
#         "train_batch_size": TRAIN_BATCH_SIZE,
#         "grad_accum_steps": GRAD_ACCUM_STEPS,
#         "num_epochs": NUM_EPOCHS,
#     }

#     temp_path = CHECKPOINT_PATH.with_suffix(".tmp")
#     torch.save(state, temp_path)
#     temp_path.replace(CHECKPOINT_PATH)


# if RESUME_IF_AVAILABLE and CHECKPOINT_PATH.exists():
#     checkpoint = torch.load(
#         CHECKPOINT_PATH,
#         map_location="cpu",
#         weights_only=False,
#     )

#     # Reject accidental resume with incompatible data/training shapes.
#     assert checkpoint["block_size"] == BLOCK_SIZE
#     assert checkpoint["train_batch_size"] == TRAIN_BATCH_SIZE
#     assert checkpoint["grad_accum_steps"] == GRAD_ACCUM_STEPS
#     assert checkpoint["precision"] == PRECISION

#     model.load_state_dict(checkpoint["model"])
#     optimizer.load_state_dict(checkpoint["optimizer"])
#     scheduler.load_state_dict(checkpoint["scheduler"])
#     scaler.load_state_dict(checkpoint["scaler"])

#     global_step = checkpoint["global_step"]
#     start_epoch = checkpoint["epoch"]
#     batch_cursor = checkpoint["batch_cursor"]
#     epoch_permutation = checkpoint["epoch_permutation"]
#     history = checkpoint["history"]

#     shuffle_generator.set_state(checkpoint["shuffle_generator"])
#     torch.set_rng_state(checkpoint["torch_rng"])
#     np.random.set_state(checkpoint["numpy_rng"])
#     random.setstate(checkpoint["python_rng"])

#     if torch.cuda.is_available() and checkpoint["cuda_rng"] is not None:
#         torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])

#     print(
#         f"\nResumed checkpoint: epoch {start_epoch + 1}, "
#         f"batch cursor {batch_cursor}, optimizer step {global_step}"
#     )
# else:
#     print("\nStarting a new training run.")


# # ------------------------------------------------------------
# # E4 / E5 / E6. TRAINING LOOP
# # ------------------------------------------------------------

# def get_training_batch(indices):
#     x = packed_data["train"]["input_ids"].index_select(0, indices)
#     y = packed_data["train"]["targets"].index_select(0, indices)
#     return x.to(device), y.to(device)


# for epoch in range(start_epoch, NUM_EPOCHS):
#     start_epoch = epoch

#     # Reuse the saved permutation when resuming partway through
#     # an epoch. Otherwise, create a deterministic new permutation.
#     if epoch_permutation is None:
#         epoch_permutation = torch.randperm(
#             train_examples,
#             generator=shuffle_generator,
#         )[:batches_per_epoch * TRAIN_BATCH_SIZE]

#         epoch_permutation = epoch_permutation.view(
#             batches_per_epoch,
#             TRAIN_BATCH_SIZE,
#         )

#         batch_cursor = 0

#     model.train()
#     optimizer.zero_grad(set_to_none=True)

#     accumulated_loss = 0.0
#     accumulated_batches = 0

#     print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

#     for batch_idx in range(batch_cursor, batches_per_epoch):
#         indices = epoch_permutation[batch_idx]
#         x, y = get_training_batch(indices)

#         with autocast_context():
#             logits = forward_logits(x)
#             loss = causal_lm_loss(logits, y)

#         if not torch.isfinite(loss):
#             raise RuntimeError(
#                 f"Non-finite training loss at step {global_step + 1}."
#             )

#         # Divide by the actual number of microbatches in this
#         # accumulation group, including a short final group.
#         group_start = (batch_idx // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
#         group_size = min(
#             GRAD_ACCUM_STEPS,
#             batches_per_epoch - group_start,
#         )

#         scaled_loss = loss / group_size

#         if USE_SCALER:
#             scaler.scale(scaled_loss).backward()
#         else:
#             scaled_loss.backward()

#         accumulated_loss += loss.item()
#         accumulated_batches += 1
#         batch_cursor = batch_idx + 1

#         is_update_step = (
#             (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
#             or batch_idx + 1 == batches_per_epoch
#         )

#         if not is_update_step:
#             continue

#         if USE_SCALER:
#             scaler.unscale_(optimizer)

#         grad_norm = torch.nn.utils.clip_grad_norm_(
#             model.parameters(),
#             MAX_GRAD_NORM,
#         )

#         if not torch.isfinite(torch.as_tensor(grad_norm)):
#             raise RuntimeError("Non-finite gradient norm.")

#         if USE_SCALER:
#             old_scale = scaler.get_scale()
#             scaler.step(optimizer)
#             scaler.update()
#             # If fp16 overflow caused a skipped optimizer update,
#             # do not advance the LR schedule or global update count.
#             optimizer_updated = scaler.get_scale() >= old_scale
#         else:
#             optimizer.step()
#             optimizer_updated = True

#         optimizer.zero_grad(set_to_none=True)

#         if optimizer_updated:
#             scheduler.step()
#             global_step += 1

#         mean_train_loss = accumulated_loss / accumulated_batches

#         history["train_steps"].append(global_step)
#         history["train_loss"].append(mean_train_loss)
#         history["lr_steps"].append(global_step)
#         history["lr"].append(optimizer.param_groups[0]["lr"])

#         accumulated_loss = 0.0
#         accumulated_batches = 0

#         if global_step % 10 == 0:
#             print(
#                 f"step {global_step:5d}/{TOTAL_STEPS} | "
#                 f"train loss {mean_train_loss:.4f} | "
#                 f"lr {optimizer.param_groups[0]['lr']:.2e}"
#             )

#         if (
#             optimizer_updated
#             and EVAL_EVERY_STEPS > 0
#             and global_step % EVAL_EVERY_STEPS == 0
#         ):
#             val_metrics = evaluate("val", MAX_EVAL_BATCHES)

#             history["val_steps"].append(global_step)
#             history["val_loss"].append(val_metrics["loss"])
#             history["val_perplexity"].append(
#                 val_metrics["perplexity"]
#             )

#             print(
#                 f"  validation loss {val_metrics['loss']:.4f} | "
#                 f"perplexity {val_metrics['perplexity']:.4f}"
#             )

#             model.train()

#         if (
#             optimizer_updated
#             and SAVE_EVERY_STEPS > 0
#             and global_step % SAVE_EVERY_STEPS == 0
#         ):
#             save_checkpoint()
#             print("  Checkpoint saved.")

#     # Epoch complete. Advance the resume position before saving.
#     start_epoch = epoch + 1
#     batch_cursor = 0
#     epoch_permutation = None

#     epoch_val = evaluate("val", MAX_EVAL_BATCHES)

#     history["val_steps"].append(global_step)
#     history["val_loss"].append(epoch_val["loss"])
#     history["val_perplexity"].append(epoch_val["perplexity"])

#     print(
#         f"Epoch {epoch + 1} complete | "
#         f"val loss {epoch_val['loss']:.4f} | "
#         f"val perplexity {epoch_val['perplexity']:.4f}"
#     )

#     save_checkpoint()
#     print("Epoch checkpoint saved.")


# # ------------------------------------------------------------
# # E6. FINAL EVALUATION + CURVES
# # ------------------------------------------------------------

# final_val = evaluate("val", max_batches=None)

# print("\nTraining complete")
# print(f"Baseline validation loss: {baseline['loss']:.4f}")
# print(f"Final validation loss:    {final_val['loss']:.4f}")
# print(f"Final validation PPL:     {final_val['perplexity']:.4f}")
# print(f"Optimizer steps:          {global_step}")
# print(f"Checkpoint:               {CHECKPOINT_PATH.resolve()}")

# plt.figure(figsize=(10, 4))
# plt.plot(
#     history["train_steps"],
#     history["train_loss"],
#     label="Training loss",
# )
# if history["val_steps"]:
#     plt.plot(
#         history["val_steps"],
#         history["val_loss"],
#         marker="o",
#         label="Validation loss",
#     )
# plt.xlabel("Optimizer step")
# plt.ylabel("Cross-entropy loss")
# plt.title("SmolLM2 Continued-Pretraining Loss")
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

# plt.figure(figsize=(9, 4))
# plt.plot(history["lr_steps"], history["lr"])
# plt.xlabel("Optimizer step")
# plt.ylabel("Learning rate")
# plt.title("Learning Rate During Training")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [32]:

import os
import json
import torch
from safetensors.torch import save_file

EXPORT_DIR = "./smollm2_hf_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Get custom model weights
state_dict = model.state_dict()

# 2. Convert parameter names to HF Llama format
hf_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("embed_tokens."):
        new_key = "model." + key

    elif key.startswith("layers."):
        new_key = "model." + key

    elif key.startswith("norm."):
        new_key = "model." + key

    elif key.startswith("lm_head."):
        new_key = key

    else:
        print("Skipping:", key)
        continue

    hf_state_dict[new_key] = value.detach().cpu().contiguous().clone()

# 3. Check tensor count and shapes
print("Original tensors:", len(state_dict))
print("Export tensors:", len(hf_state_dict))

# 4. Save weights
save_file(
    hf_state_dict,
    os.path.join(EXPORT_DIR, "model.safetensors")
)

print("Weights exported successfully!")


Original tensors: 273
Export tensors: 273
Weights exported successfully!


In [33]:

config_dict = {
    "architectures": ["LlamaForCausalLM"],
    "model_type": "llama",
    "hidden_size": 576,
    "intermediate_size": 1536,
    "num_hidden_layers": 30,
    "num_attention_heads": 9,
    "num_key_value_heads": 3,
    "max_position_embeddings": 8192,
    "rms_norm_eps": 1e-5,
    "rope_theta": 100000.0,
    "hidden_act": "silu",
    "vocab_size": 49152,
    "tie_word_embeddings": True,
    "attention_bias": False,
    "mlp_bias": False,
    "attention_dropout": 0.0,
    "rope_interleaved": False,
    "bos_token_id": tokenizer.bos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
    "torch_dtype": "float32",
    "use_cache": True
}

with open(
    os.path.join(EXPORT_DIR, "config.json"), "w"
) as f:
    json.dump(config_dict, f, indent=2)

print("Config exported!")


Config exported!


In [34]:
tokenizer.save_pretrained(EXPORT_DIR)

from transformers import AutoModelForCausalLM

hf_exported_model = AutoModelForCausalLM.from_pretrained(
    EXPORT_DIR,
    torch_dtype=torch.float32
)

hf_exported_model.eval()

print(hf_exported_model)

Loading weights: 100%|██████████| 273/273 [00:00<00:00, 1027.31it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [35]:
python convert_hf_to_gguf.py ../smollm2_hf_export \
    --outfile ../smollm2-trained-f16.gguf \
    --outtype f16

SyntaxError: invalid syntax (1813451744.py, line 1)